In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import torch
from torch.utils.data import DataLoader
from helper.golbal_variables import *
from finetune.cnn14 import Cnn14
from finetune.modelpipeline import ModelPipeline
from finetune.pred import predict, Mashup_Chunk, collate_fn_mashup

In [2]:
# import os 
# import requests
# from pathlib import Path

# Path("pretrained_models").mkdir(exist_ok=True)

# url = "https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth"

# output_path = "pretrained_models/Cnn14.pth"

# with requests.get(url, stream=True) as r:
#     r.raise_for_status()
#     with open(output_path, "wb") as f:
#         for chunk in r.iter_content(chunk_size=8192):
#             f.write(chunk)

# print("Download complete")

In [3]:
# model_pipeline = ModelPipeline(
#     esc50_dir=ESC_50_MASTER,
#     genre_dir=GENRES_STEMS
# )

# model_pipeline.complete_training()

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
model = Cnn14(classes_num=10).to(device)

checkpoint = torch.load("Z:\\dlgen-AI-project\\finetune\\best_genre_model.pth", map_location=device)

if isinstance(checkpoint, dict) and "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint
        
model.load_state_dict(state_dict, strict=False)

<All keys matched successfully>

In [4]:
genre_map = {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}

In [5]:
test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv")
output = {}

for idx, row in test_df.iterrows():
    song_id = row["id"]
    filename = row["filename"]
    
    audio_path = MESSY_DATA_DIR / filename
    result = predict(
        model=model,
        audio_path=audio_path,
        device=device,
        genre_map=genre_map
    )

    output[song_id] = result
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} songs...")

output_df = pd.DataFrame(list(output.items()), columns=['id', 'genre'])
output_df.to_csv("output.csv")

Processed 100/3020 songs...
Processed 200/3020 songs...
Processed 300/3020 songs...
Processed 400/3020 songs...
Processed 500/3020 songs...
Processed 600/3020 songs...
Processed 700/3020 songs...
Processed 800/3020 songs...
Processed 900/3020 songs...
Processed 1000/3020 songs...
Processed 1100/3020 songs...
Processed 1200/3020 songs...
Processed 1300/3020 songs...
Processed 1400/3020 songs...
Processed 1500/3020 songs...
Processed 1600/3020 songs...
Processed 1700/3020 songs...
Processed 1800/3020 songs...
Processed 1900/3020 songs...
Processed 2000/3020 songs...
Processed 2100/3020 songs...
Processed 2200/3020 songs...
Processed 2300/3020 songs...
Processed 2400/3020 songs...
Processed 2500/3020 songs...
Processed 2600/3020 songs...
Processed 2700/3020 songs...
Processed 2800/3020 songs...
Processed 2900/3020 songs...
Processed 3000/3020 songs...


In [9]:
output_df = output_df.set_index('id')

In [7]:
import pandas as pd 
df = pd.read_csv("output.csv")

In [8]:
df["genre"].unique()

<StringArray>
['pop', 'hiphop', 'metal', 'country', 'jazz', 'classical', 'reggae']
Length: 7, dtype: str

In [10]:
output_df.to_csv("output.csv")